In [ ]:
import sys
import os
from dotenv import load_dotenv
load_dotenv()

sys.path.append(".")
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots
import comet_ml
from dataclasses import dataclass, asdict
import plotly.express as px


# Setup

In [ ]:
plt.style.use(["science", "ieee", "grid"])

In [ ]:
COMET_API_KEY = os.environ.get("COMET_API_KEY")
comet_ml.login(api_key=COMET_API_KEY)

In [ ]:
api = comet_ml.API()

# Load experiment data

In [ ]:
all_experiments_in_project = api.get_experiments(
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name=os.environ.get("COMET_PROJECT_VAE"),
)

In [ ]:
params_to_avoid = [
    "batch_size",
    "learning_rate",
    "activation",
    "curr_epoch",
    "curr_step",
    "normalize",
    "num_layers",
    "num_epochs",
]

metrics_to_avoid = [
    "sys.compute.overall",
    "sys.compute.utilized",
    "sys.cpu.percent.avg",
    "sys.disk.read_bps",
    "sys.disk.write_bps",
    "sys.gpu.0.free_memory",
    "sys.gpu.0.gpu_utilization",
    "sys.gpu.0.memory_utilization",
    "sys.gpu.0.percent.used_memory",
    "sys.gpu.0.power_usage",
    "sys.gpu.0.temperature",
    "sys.gpu.0.total_memory",
    "sys.gpu.0.used_memory",
    "sys.load.avg",
    "sys.network.receive_bps",
    "sys.network.send_bps",
    "sys.ram.available",
    "sys.ram.percent.used",
    "sys.ram.total",
    "sys.ram.used",
]

In [ ]:
@dataclass
class Parameter:
    experiment_id: str
    name: str
    value: str


@dataclass
class Metric:
    experiment_id: str
    name: str
    value: float

In [ ]:
parameters = []
metrics = []
for experiment in all_experiments_in_project:
    name = experiment.get_name()
    params = experiment.get_parameters_summary()
    raw_metrics = experiment.get_metrics()
    for param in params:
        if param.get("name") in params_to_avoid:
            continue
        parameters.append(
            Parameter(
                experiment_id=name, name=param.get("name"), value=param.get("valueMax")
            )
        )
    for metric_dict in raw_metrics:
        if (
            type(metric_dict) != dict
            or metric_dict.get("metricName") in metrics_to_avoid
        ):
            continue
        metrics.append(
            Metric(
                experiment_id=name,
                name=metric_dict.get("metricName"),
                value=float(metric_dict.get("metricValue")),
            )
        )

# Process metrics

In [ ]:
experiments_to_avoid = []

In [ ]:
metrics_df = pd.DataFrame(
    [asdict(m) for m in metrics if m.experiment_id not in experiments_to_avoid]
).pivot_table(index="experiment_id", columns="name", values="value", aggfunc="first")

params_df = pd.DataFrame(
    [asdict(p) for p in parameters if p.experiment_id not in experiments_to_avoid]
).pivot_table(index="experiment_id", columns="name", values="value", aggfunc="first")

In [ ]:
missing_id = set(params_df.index) - set(metrics_df.index)

In [ ]:
missing_df = params_df.loc[params_df.index.isin(missing_id)]

In [ ]:
df = pd.merge(metrics_df, params_df, left_index=True, right_index=True)

In [ ]:
df["avg_val_pearson"] = df.apply(
    lambda row: np.mean(
        [
            abs(row["valid_0_vs_1_pearson"]),
            abs(row["valid_0_vs_2_pearson"]),
            abs(row["valid_0_vs_3_pearson"]),
            abs(row["valid_1_vs_2_pearson"]),
            abs(row["valid_1_vs_3_pearson"]),
            abs(row["valid_2_vs_3_pearson"]),
        ]
    ),
    axis=1,
)

df["min_val_pearson"] = df.apply(
    lambda row: np.min(
        [
            abs(row["valid_0_vs_1_pearson"]),
            abs(row["valid_0_vs_2_pearson"]),
            abs(row["valid_0_vs_3_pearson"]),
            abs(row["valid_1_vs_2_pearson"]),
            abs(row["valid_1_vs_3_pearson"]),
            abs(row["valid_2_vs_3_pearson"]),
        ]
    ),
    axis=1,
)

df["max_val_pearson"] = df.apply(
    lambda row: np.max(
        [
            abs(row["valid_0_vs_1_pearson"]),
            abs(row["valid_0_vs_2_pearson"]),
            abs(row["valid_0_vs_3_pearson"]),
            abs(row["valid_1_vs_2_pearson"]),
            abs(row["valid_1_vs_3_pearson"]),
            abs(row["valid_2_vs_3_pearson"]),
        ]
    ),
    axis=1,
)

df["avg_train_pearson"] = df.apply(
    lambda row: np.mean(
        [
            abs(row["train_0_vs_1_pearson"]),
            abs(row["train_0_vs_2_pearson"]),
            abs(row["train_0_vs_3_pearson"]),
            abs(row["train_1_vs_2_pearson"]),
            abs(row["train_1_vs_3_pearson"]),
            abs(row["train_2_vs_3_pearson"]),
        ]
    ),
    axis=1,
)

In [ ]:
df["kl_weight"] = df["kl_weight"].astype(float)
df["kl_weight_log10"] = np.log10(df.kl_weight)
df["val_kl_loss_log10"] = np.log10(df.val_kl_loss)
df["train_kl_loss_log10"] = np.log10(df.train_kl_loss)

df["latent_dim"] = df["latent_dim"].astype(int)

In [ ]:
df["val_dist_loss_norm"] = df["val_dist_loss"] / (df["latent_dim"])
df["train_dist_loss_norm"] = df["train_dist_loss"] / (df["latent_dim"])


# Analysis

In [ ]:
df.loc["colorful_moss_1642"]

In [ ]:
candidates = df.loc[df["val_kl_loss_log10"] > 0]
best = candidates["val_rec_loss"].argmin()
candidates.iloc[best]

In [ ]:
px.histogram(
    df, x="val_rec_loss", nbins=30, facet_row="max_level", height=600, width=400
)

In [ ]:
import matplotlib.pyplot as plt

# Ensure max_level is integer for correct sorting/grouping
df_plot = df.copy()
df_plot["max_level"] = df_plot["max_level"].astype(int)

max_levels = sorted(df_plot["max_level"].unique())
n_levels = len(max_levels)

fig, axes = plt.subplots(n_levels, 1, figsize=(2.3, 5), sharex=True)

for i, level in enumerate(max_levels):
    ax = axes[i] if n_levels > 1 else axes
    vals_valid = df_plot[df_plot["max_level"] == level]["val_rec_loss"]
    # vals_train = df_plot[df_plot["max_level"] == level]["train_rec_loss"]
    ax.hist(vals_valid, bins=15, color="skyblue", edgecolor="black")
    # ax.hist(vals_train, bins=15, color="red", edgecolor="black")
    ax.set_ylabel("Count")
    ax.set_title(f"WPT maximum level: {level}")
    # if i == 0:
    #     ax.set_title("val_rec_loss Histogram by max_level")
    if i < n_levels - 1:
        ax.set_xlabel("")
    else:
        ax.set_xlabel("Validation reconstruction loss")

plt.tight_layout()
# plt.show()
plt.savefig("val_rec_loss_histogram_by_max_level.png", dpi=600)

In [ ]:
temp = df.copy()
temp["kl_weight_log10"] = temp["kl_weight_log10"].astype(int)

px.histogram(
    temp.sort_values("kl_weight_log10"),
    x="val_kl_loss_log10",
    facet_row="kl_weight_log10",
    nbins=20,
    height=700,
    width=400,
)

In [ ]:
from matplotlib.ticker import FuncFormatter


temp = df.copy()
temp["kl_weight_log10"] = temp["kl_weight_log10"].astype(int)
kl_weights = sorted(temp["kl_weight_log10"].unique())
n_weights = len(kl_weights)

fig, axes = plt.subplots(n_weights, 1, figsize=(2.3, 6), sharex=True)

for i, kw in enumerate(kl_weights):
    ax = axes[i] if n_weights > 1 else axes
    vals = temp[temp["kl_weight_log10"] == kw]["val_kl_loss_log10"]
    ax.hist(vals, bins=10, color="skyblue", edgecolor="black")
    ax.set_ylabel("Count")
    ax.set_title(r"$\beta$: $10^{{{}}}$".format(kw))
    if i < n_weights - 1:
        ax.set_xlabel("")
    else:
        ax.set_xlabel("Validation KL loss")
# Replace x labels by 10^original label


def exp_fmt(x, pos):
    return r"$10^{{{:.0f}}}$".format(x)


ax = axes if n_weights == 1 else None
for i, kw in enumerate(kl_weights):
    ax = axes[i] if n_weights > 1 else axes
    ax.xaxis.set_major_formatter(FuncFormatter(exp_fmt))

plt.tight_layout()
plt.savefig("val_kl_loss_log10_hist_by_kl_weight.png", dpi=600)

In [ ]:
temp = df.copy()
temp["latent_dim"] = temp["latent_dim"].astype(int)

px.histogram(
    temp.sort_values("latent_dim"),
    x="val_dist_loss_norm",
    facet_row="latent_dim",
    # facet_col="beta",
    nbins=20,
    height=700,
    width=800,
)

In [ ]:
temp = df.copy()
temp["latent_dim"] = temp["latent_dim"].astype(int)

px.box(
    temp.sort_values("latent_dim"),
    x="val_dist_loss_norm",
    facet_row="latent_dim",
    # facet_col="beta",
    #nbins=20,
    height=700,
    width=800,
)

In [ ]:
temp = df.copy()
temp["latent_dim"] = temp["latent_dim"].astype(int)

px.histogram(
    temp.sort_values("latent_dim"),
    x="val_dist_loss_norm",
    facet_row="beta",
    # facet_col="beta",
    nbins=20,
    height=700,
    width=800,
)

In [ ]:
temp = df.copy()
temp["latent_dim"] = temp["latent_dim"].astype(int)
latent_dims = sorted(temp["latent_dim"].unique())
n_dims = len(latent_dims)

fig, axes = plt.subplots(n_dims, 1, figsize=(2, 5), sharex=True)

for i, ld in enumerate(latent_dims):
    ax = axes[i] if n_dims > 1 else axes
    vals = temp[temp["latent_dim"] == ld]["val_dist_loss_norm"]
    # ax.hist(vals, bins=15, color="salmon", edgecolor="black")
    ax.hist(vals, bins=10, color="skyblue", edgecolor="black")
    ax.set_ylabel("Count")
    # ax.set_title(f"Latent dim: {ld}")
    ax.set_title(r"$N_z$: ${}$".format(ld))
    if i < n_dims - 1:
        ax.set_xlabel("")
    else:
        ax.set_xlabel("Validation distance loss")

plt.tight_layout()
plt.savefig("val_dist_loss_hist_by_latent_dim.png", dpi=600)

In [ ]:
temp = df.copy()
temp["latent_dim"] = temp["latent_dim"].astype(int)
betas = sorted(temp["beta"].unique())
n_dims = len(betas)

fig, axes = plt.subplots(n_dims, 1, figsize=(2, 4), sharex=True)

for i, ld in enumerate(betas):
    ax = axes[i] if n_dims > 1 else axes
    vals = temp[temp["beta"] == ld]["val_dist_loss_norm"]
    # ax.hist(vals, bins=15, color="salmon", edgecolor="black")
    ax.hist(vals, bins=10, color="skyblue", edgecolor="black")
    ax.set_ylabel("Count")
    # ax.set_title(r"$\alpha$: {ld}")
    ax.set_title(r"$\alpha$: ${}$".format(ld))
    if i < n_dims - 1:
        ax.set_xlabel("")
    else:
        ax.set_xlabel("Validation distance loss")

plt.tight_layout()
plt.savefig("val_dist_loss_hist_by_alpha.png", dpi=600)

In [ ]:
df.sort_values("avg_train_pearson", ascending=False)[
    ["avg_train_pearson", "val_kl_loss"]
]

In [ ]:
df["val_full_loss"] = df["val_dist_loss"] + df["val_kl_loss"] + df["val_rec_loss"]
df["val_dist_percent"] = df["val_dist_loss"] / df["val_full_loss"]
df["val_rec_percent"] = df["val_rec_loss"] / df["val_full_loss"]
df["val_kl_percent"] = df["val_kl_loss"] / df["val_full_loss"]

In [ ]:
df.loc["colorful_moss_1642"]["kl_weight_log10"]

In [ ]:
px.histogram(
    df.sort_values("kl_weight_log10"),
    x="avg_val_pearson",
    facet_row="kl_weight_log10",
    color="beta",
    # color="latent_dim",
    nbins=15,
    height=1100,
    width=500,
)

In [ ]:
df.loc["colorful_moss_1642"]["avg_val_pearson"]

In [ ]:
df.sort_values(["val_rec_loss"])[
    ["val_rec_loss", "val_kl_loss_log10", "val_dist_loss_norm"]
]

In [ ]:
df.loc[df["kl_weight_log10"] < 0].sort_values(["val_kl_loss"])[
    ["val_rec_loss", "val_kl_loss_log10", "val_dist_loss_norm", "kl_weight_log10"]
]

In [ ]:
df.sort_values(["val_dist_loss_norm"])[
    ["val_rec_loss", "val_kl_loss_log10", "val_dist_loss_norm", "kl_weight_log10"]
]

In [ ]:
temp = df.copy()
temp = temp.sort_values("latent_dim")
temp["latent_dim"] = temp["latent_dim"].astype(str)

px.scatter(
    temp.reset_index(),
    y="val_dist_loss_norm",
    x="val_kl_loss_log10",
    hover_data=["experiment_id"],
    color="latent_dim",
)

In [ ]:
temp = df.copy()
temp = temp.sort_values("latent_dim")
temp["latent_dim"] = temp["latent_dim"].astype(str)

fig, ax = plt.subplots(figsize=(4, 2))
groups = temp.groupby("latent_dim")

markers = ["o", "x", ">", "d"]
for i, (name, group) in enumerate(groups):
    ax.scatter(
        group["val_kl_loss_log10"],
        group["val_dist_loss_norm"],
        label=f"latent_dim {name}",
        alpha=0.7,
        s=10,
        marker=markers[i],
    )

ax.set_xlabel("Validation KL loss")
ax.set_ylabel("Validation distance loss")
ax.legend(title="$N_z$", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Correlation

In [ ]:
labels = ["low", "medium", "high", "very high"]

df["val_rec_quantile"] = pd.qcut(
    df["val_rec_loss"], 4, labels=labels, duplicates="drop"
)
df["val_kl_quantile"] = pd.qcut(
    df["val_kl_loss_log10"], 4, labels=labels, duplicates="drop"
)
df["val_dist_quantile"] = pd.qcut(
    df["val_dist_loss_norm"], 4, labels=labels, duplicates="drop"
)
df["val_pearson_quantile"] = pd.qcut(
    df["avg_val_pearson"], 4, labels=labels, duplicates="drop"
)

In [ ]:
px.line(
    df.sort_values("kl_weight_log10", ascending=True),
    x="kl_weight_log10",
    y="avg_val_pearson",
    facet_row="latent_dim",
    facet_col="max_level",
    color="beta",
    width=800,
    height=600,
)

In [ ]:
sorted_df = df.sort_values("kl_weight_log10", ascending=True)

latent_dims = np.sort(sorted_df["latent_dim"].unique())
max_levels = np.sort(sorted_df["max_level"].unique())
betas = sorted_df["beta"].unique()

n_rows = len(latent_dims)
n_cols = len(max_levels)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 8), sharex=True, sharey=True)

# If there's only one row or column, axes may not be a 2D array. Force it to 2D.
if n_rows == 1 and n_cols == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes[np.newaxis, :]
elif n_cols == 1:
    axes = axes[:, np.newaxis]

markers = ["o", "x", ">", "d"]
for i, latent_dim in enumerate(latent_dims):
    for j, max_level in enumerate(max_levels):
        ax = axes[i, j]
        sub_df = sorted_df[
            (sorted_df["latent_dim"] == latent_dim)
            & (sorted_df["max_level"] == max_level)
        ]
        if sub_df.empty:
            ax.axis("off")
            continue
        beta_vals = sub_df["beta"].sort_values().unique()
        for k, beta in enumerate(beta_vals):
            beta_df = sub_df[sub_df["beta"] == beta]
            ax.plot(
                beta_df["kl_weight_log10"],
                beta_df["avg_val_pearson"],
                marker=markers[k],
                label=f"beta={beta}",
            )
        ax.set_title(r"$N_z=$" + f"{latent_dim}, depth={max_level}")
        ax.set_xlabel(r"$\beta$")
        ax.set_xticks(beta_df["kl_weight_log10"])
        # ax.legend()
        ax.set_ylabel("Pearson Correlation")

        if i == 0 and j == 2:
            legend_labels = [r"$\alpha=$" + f"{beta}" for beta in beta_vals]
            ax.legend(legend_labels, bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("val_pearson_by_hyperparameters.png", dpi=600)
plt.show()

In [ ]:
df.columns

In [ ]:
px.scatter(df, x="valid_0_vs_1_pearson", y="valid_0_vs_2_pearson")

In [ ]:
px.histogram(
    df.sort_values("val_rec_loss"),
    x="avg_val_pearson",
    facet_row="val_rec_quantile",
    # color="val_rec_quantile",
    nbins=10,
    width=600,
    height=800,
)

In [ ]:
df["delta_val_pearson"] = df["max_val_pearson"] - df["min_val_pearson"]

In [ ]:
correlations_raw = df[
    [
        "valid_0_vs_1_pearson",
        "valid_0_vs_2_pearson",
        "valid_0_vs_3_pearson",
        "valid_1_vs_2_pearson",
        "valid_1_vs_3_pearson",
        "valid_2_vs_3_pearson",
        "kl_weight_log10",
        "latent_dim",
        "max_level",
        "beta",
        "val_rec_quantile",
        "val_kl_quantile",
        "val_dist_quantile",
    ]
]

In [ ]:
correlations = pd.melt(
    correlations_raw,
    value_vars=[
        "valid_0_vs_1_pearson",
        "valid_0_vs_2_pearson",
        "valid_0_vs_3_pearson",
        "valid_1_vs_2_pearson",
        "valid_1_vs_3_pearson",
        "valid_2_vs_3_pearson",
    ],
    id_vars=[
        "kl_weight_log10",
        "latent_dim",
        "max_level",
        "beta",
        "val_rec_quantile",
        "val_kl_quantile",
        "val_dist_quantile",
    ],
)

In [ ]:
correlations.head(3)

In [ ]:
# Get unique correlation names and count
cor_names = correlations['name'].unique()
n_rows = len(cor_names)
fig, axes = plt.subplots(
    3, 2, figsize=(3, 3), sharex=False
)

if n_rows == 1:
    axes = [axes]

cor_name_mapper = {
    "valid_0_vs_1_pearson": "$r_{1,2}$",
    "valid_0_vs_2_pearson": "$r_{1,3}$",
    "valid_0_vs_3_pearson": "$r_{1,4}$",
    "valid_1_vs_2_pearson": "$r_{2,3}$",
    "valid_1_vs_3_pearson": "$r_{2,4}$",
    "valid_2_vs_3_pearson": "$r_{3,4}$",
}

flat_axes = axes.flatten()
for i, cor_name in enumerate(cor_names):
    subset = correlations[correlations["name"] == cor_name]
    values = subset["value"]
    median = np.median(values)
    flat_axes[i].hist(values, bins=15, color="skyblue", edgecolor="grey")
    flat_axes[i].axvline(median, color="black", linestyle="--", linewidth=1.5)
    flat_axes[i].set_ylabel("Count")
    flat_axes[i].set_title(cor_name_mapper[cor_name])
    # flat_axes[i].legend(["Median"], fontsize=5, loc="upper left")
    # flat_axes[i].text(median, -6, f"{median:.2f}", fontsize=5, ha="center")
    flat_axes[i].set_xlim(-1.1, 1.1)
    flat_axes[i].set_xticks([-1, 0, median, 1])
    flat_axes[i].set_xticklabels([str(-1), str(0), str(round(median,2)), str(1)], fontsize=6)

axes[2, 1].set_xlabel("Pearson correlation")
axes[2, 0].set_xlabel("Pearson correlation")


plt.tight_layout()
plt.savefig("val_pearson_hist.png", dpi=600)
plt.show()

In [ ]:


cor_name_mapper = {
    "valid_0_vs_1_pearson": "$r_{1,2}$",
    "valid_0_vs_2_pearson": "$r_{1,3}$",
    "valid_0_vs_3_pearson": "$r_{1,4}$",
    "valid_1_vs_2_pearson": "$r_{2,3}$",
    "valid_1_vs_3_pearson": "$r_{2,4}$",
    "valid_2_vs_3_pearson": "$r_{3,4}$",
}

# Calculate medians and prepare dataset tuples: (median, label, values)
grouped_data = []
for cor_name in correlations["name"].unique():
    vals = correlations[correlations["name"] == cor_name]["value"].values
    med = np.median(vals)
    label = cor_name_mapper.get(cor_name, cor_name)
    grouped_data.append((med, label, vals))

# Sort groups in descending order based on median
grouped_data.sort(key=lambda item: item[0], reverse=True)

# Extract sorted values and labels
data_to_plot = [item[2] for item in grouped_data]
labels = [item[1] for item in grouped_data]

# Plot vertical boxplots
fig, ax = plt.subplots(figsize=(5, 3))

ax.boxplot(data_to_plot, labels=labels, vert=True)

ax.set_xlabel("Cable Pair")
ax.set_ylabel("Pearson correlation")
ax.set_ylim(-1.1, 1.1)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
# plt.savefig("val_pearson_boxplot_sorted.png", dpi=600)
plt.show()

In [ ]:

cor_name_mapper = {
    "valid_0_vs_1_pearson": "$r_{1,2}$",
    "valid_0_vs_2_pearson": "$r_{1,3}$",
    "valid_0_vs_3_pearson": "$r_{1,4}$",
    "valid_1_vs_2_pearson": "$r_{2,3}$",
    "valid_1_vs_3_pearson": "$r_{2,4}$",
    "valid_2_vs_3_pearson": "$r_{3,4}$",
}

# Calculate medians and prepare dataset tuples: (median, label, values)
grouped_data = []
for cor_name in correlations["name"].unique():
    vals = correlations[correlations["name"] == cor_name]["value"].values
    med = np.median(vals)
    label = cor_name_mapper.get(cor_name, cor_name)
    grouped_data.append((med, label, vals))

# Sort ascending so the largest median appears at the top of the y-axis
grouped_data.sort(key=lambda item: item[0], reverse=False)

# Extract sorted values and labels
data_to_plot = [item[2] for item in grouped_data]
labels = [item[1] for item in grouped_data]

# Plot horizontal boxplots (transposed)
fig, ax = plt.subplots(figsize=(3, 3))

ax.boxplot(data_to_plot, tick_labels=labels, vert=False, showfliers=False, )

ax.set_xlabel("Pearson correlation")
ax.set_ylabel("Cable Pair")
ax.set_xlim(-1.1, 1.1)
# ax.grid(True, linestyle="--", alpha=0.5)
ax.grid(True, axis="x", which="major", linestyle="--", alpha=0.35)
# ax.grid(False)
plt.tight_layout()
plt.savefig("figures_hyperparameters/val_pearson_boxplot_transposed.png", dpi=600)
plt.show()

In [ ]:
temp = df.loc[(df["kl_weight_log10"] < 0) & (df["val_rec_quantile"] == "low")].copy()

temp.sort_values("min_val_pearson", ascending=False)[
    [
        "min_val_pearson",
        "avg_val_pearson",
        "max_val_pearson",
        "kl_weight_log10",
        "latent_dim",
        "max_level",
        "beta",
        "val_rec_quantile",
        "val_kl_quantile",
        "val_dist_quantile",
        "valid_0_vs_1_pearson",
        "valid_0_vs_2_pearson",
        "valid_0_vs_3_pearson",
        "valid_1_vs_2_pearson",
        "valid_1_vs_3_pearson",
        "valid_2_vs_3_pearson",
    ]
].head(20)